# Track 02.1 — Dense Embeddings, Vector Spaces, and Distance Metrics

> **RAG Engineering Academy — Track 02: Embeddings & Vector Search**  
> *Core Focus: Geometric Foundations of Embedding Spaces, Distance & Similarity Metrics, $L_2$ Normalization Invariants, Dimensional Collapse, and the Curse of Dimensionality.*

---

## 🎯 Executive Overview & Pedagogical Goals

In modern Retrieval-Augmented Generation (RAG) systems, raw text must be converted into numerical representations that capture semantic meaning. While keyword methods (such as BM25) match exact lexical tokens, **dense embeddings** map words, sentences, and documents into continuous vector spaces $\mathbb{R}^D$ where geometric proximity reflects conceptual similarity:

$$\text{Semantic Similarity} \iff \text{Geometric Proximity in } \mathbb{R}^D$$

However, vector spaces in high dimensions ($D \in [384, 1536]$) do not behave like familiar 2D or 3D Euclidean space. Without understanding the underlying geometry, engineers encounter subtle bugs:
- Picking the wrong distance metric (e.g., using unnormalized Dot Product for variable-length chunks).
- Assuming high-dimensional Euclidean distance provides fine-grained contrast when distances suffer from **concentration of measure**.
- Overlooking **dimensional collapse (anisotropy)** in transformer representations, leading to high baseline cosine similarity even for unrelated text.

### Key Learning Objectives
1. **The Vector Space Model:** Understand coordinates, magnitudes ($\|\mathbf{u}\|_2$), directions ($\hat{\mathbf{u}}$), and angular separation in continuous semantic vector space.
2. **The Metric Trinity from First Principles:** Implement Dot Product, Cosine Similarity/Distance, and Euclidean ($L_2$) Distance from scratch in pure NumPy.
3. **The Fundamental Equivalence Theorem:** Prove algebraically and numerically why Cosine Similarity, Dot Product, and Euclidean Distance yield identical rankings on the unit hypersphere $\mathbb{S}^{D-1}$.
4. **Vectorized Batch Retrieval:** Implement hardware-friendly matrix-vector retrieval (`GEMM`), calculating top-$k$ candidate ranking over chunk corpora.
5. **The Curse of Dimensionality:** Simulate and visualize distance concentration and geometric orthogonality in high-dimensional spaces ($D \to 1536$).
6. **Dimensional Collapse & Anisotropy:** Analyze the "narrow cone problem" in transformer embeddings using Singular Value Decomposition (SVD) and implement post-processing mitigations (mean-centering).

---

## 🏗️ Architecture: From Text to Geometric Vector Space

```mermaid
flowchart TD
    Raw["Raw Text Chunk: 'Neural networks learn distributed representations'"] --> Tokenizer["Subword Tokenizer (BPE / WordPiece)"]
    Tokenizer --> Transformer["Transformer Encoder (e.g., MiniLM, BERT, GTE)"]
    Transformer --> Hidden["Sequence Hidden States: H in R^(L x D)"]
    Hidden --> Pooling["Mean Pooling across Token Dimension: u in R^D"]
    Pooling --> L2Norm["L2 Normalization: u_hat = u / ||u||_2"]
    L2Norm --> Sphere["Unit Hypersphere S^(D-1) in R^D"]
    Sphere --> Distance["Geometric Distance Comparison (Dot / Cosine / L2)"]

    classDef stage fill:#f8fafc,stroke:#3b82f6,stroke-width:1.5px,color:#1e293b;
    class Raw,Tokenizer,Transformer,Hidden,Pooling,L2Norm,Sphere,Distance stage;
```

## 0. Environment Setup & Configuration

We establish our Python execution environment with zero external model downloads or cloud API dependencies. All linear algebra operations are computed using **NumPy**, and geometric distributions are visualized using **Matplotlib**.

We set explicit random seeds to ensure 100% deterministic reproducibility across runs.

In [ ]:
import os
import sys
import math
import numpy as np
import matplotlib
matplotlib.use("Agg")  # Non-interactive headless backend for seamless headless/CI execution
import matplotlib.pyplot as plt
from typing import Tuple, List, Dict, Optional

# Ensure matplotlib runs headless and cleanly without GUI dependencies
os.environ["MPLCONFIGDIR"] = "/tmp/matplotlib-cache"
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 10

# Set deterministic random seed for reproducible simulations
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print(f"NumPy version: {np.__version__}")
print(f"Python interpreter: {sys.executable}")
print("Offline environment configured successfully.")

## 1. Vector Space Geometry: Coordinates, Magnitudes, and Angles

A dense embedding represents a piece of text as a point (or directed arrow from the origin) in a $D$-dimensional real coordinate space $\mathbb{R}^D$:

$$\mathbf{u} = \begin{bmatrix} u_1 \\ u_2 \\ \vdots \\ u_D \end{bmatrix} \in \mathbb{R}^D$$

Every vector in this space possesses two fundamental geometric properties:
1. **Magnitude (Length / $L_2$ Norm):** The Euclidean distance from the origin to the vector tip:
   $$\|\mathbf{u}\|_2 = \sqrt{\sum_{i=1}^D u_i^2} = \sqrt{\mathbf{u}^T \mathbf{u}}$$
2. **Direction (Unit Vector):** The pure angular orientation of the vector, obtained by dividing the vector by its magnitude:
   $$\hat{\mathbf{u}} = \frac{\mathbf{u}}{\|\mathbf{u}\|_2}, \quad \text{such that } \|\hat{\mathbf{u}}\|_2 = 1.0$$
   All unit vectors lie on the surface of the **unit hypersphere** $\mathbb{S}^{D-1}$.

The geometric angle $\theta$ between two vectors $\mathbf{u}$ and $\mathbf{v}$ is given by the cosine formula:
$$\cos\theta = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} = \hat{\mathbf{u}} \cdot \hat{\mathbf{v}}$$

In the following code cell, we construct sample vectors in 2D space, calculate their magnitudes, compute their unit directions, and determine their mutual angle.

In [ ]:
# Define two sample vectors in R^2 for intuitive visual exploration
u = np.array([3.0, 4.0], dtype=np.float64)   # Length 5.0 (3-4-5 right triangle)
v = np.array([1.0, 3.0], dtype=np.float64)   # Length sqrt(10) ≈ 3.162

# 1. Magnitude (L2 norm) from first principles vs np.linalg.norm
norm_u_manual = math.sqrt(sum(x ** 2 for x in u))
norm_u_np = float(np.linalg.norm(u))
assert np.isclose(norm_u_manual, norm_u_np), "Norm computation mismatch!"

# 2. Unit vector normalization
u_hat = u / norm_u_np
v_hat = v / np.linalg.norm(v)

# 3. Inner product and Angular separation
dot_product = float(np.dot(u, v))
cos_theta = float(np.dot(u_hat, v_hat))
theta_radians = math.acos(np.clip(cos_theta, -1.0, 1.0))
theta_degrees = math.degrees(theta_radians)

print(f"Vector u: {u} | Magnitude ||u||: {norm_u_np:.4f} | Unit u_hat: {u_hat}")
print(f"Vector v: {v} | Magnitude ||v||: {np.linalg.norm(v):.4f} | Unit v_hat: {v_hat}")
print(f"Inner Product <u, v>: {dot_product:.4f}")
print(f"Cosine of Angle (cos θ): {cos_theta:.4f}")
print(f"Angle θ: {theta_radians:.4f} rad ({theta_degrees:.2f}°)")

### Visualizing 2D Vector Geometry and Unit Projection

To make the abstract linear algebra tangible, the next cell generates a geometric diagram illustrating:
- The raw vectors $\mathbf{u}$ and $\mathbf{v}$ with their respective magnitudes.
- The unit circle $\mathbb{S}^1$ where normalized vectors lie.
- The normalized unit directions $\hat{\mathbf{u}}$ and $\hat{\mathbf{v}}$.
- The angular separation $\theta$.
- The orthogonal projection of $\mathbf{u}$ onto $\mathbf{v}$ demonstrating the geometric meaning of the inner product.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))

# Draw unit circle
theta_circle = np.linspace(0, 2 * np.pi, 200)
ax.plot(np.cos(theta_circle), np.sin(theta_circle), 'k--', alpha=0.35, label="Unit Circle ($||x|| = 1$)")

# Plot raw vectors u and v
ax.quiver(0, 0, u[0], u[1], angles='xy', scale_units='xy', scale=1, color='#2563eb', label=r'Raw $\mathbf{u} = [3, 4]$ ($||\mathbf{u}||=5.0$)')
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1, color='#dc2626', label=r'Raw $\mathbf{v} = [1, 3]$ ($||\mathbf{v}||\approx 3.16$)')

# Plot unit vectors u_hat and v_hat
ax.quiver(0, 0, u_hat[0], u_hat[1], angles='xy', scale_units='xy', scale=1, color='#3b82f6', linestyle=':', label=r'Unit $\hat{\mathbf{u}}$ ($||\hat{\mathbf{u}}||=1.0$)')
ax.quiver(0, 0, v_hat[0], v_hat[1], angles='xy', scale_units='xy', scale=1, color='#ef4444', linestyle=':', label=r'Unit $\hat{\mathbf{v}}$ ($||\hat{\mathbf{v}}||=1.0$)')

# Plot orthogonal projection of u onto v
proj_u_on_v = (np.dot(u, v) / np.dot(v, v)) * v
ax.plot([u[0], proj_u_on_v[0]], [u[1], proj_u_on_v[1]], 'gray', linestyle='--', alpha=0.7, label=r'Orthogonal projection')

# Annotation of angle
ax.text(0.6, 1.2, f"θ = {theta_degrees:.1f}°\ncos(θ) = {cos_theta:.3f}", fontsize=11, fontweight='semibold',
        bbox=dict(boxstyle="round,pad=0.3", fc="#f8fafc", ec="#cbd5e1", lw=1))

ax.set_xlim(-1.5, 5.5)
ax.set_ylim(-1.5, 5.5)
ax.axhline(0, color='gray', linewidth=0.8, alpha=0.5)
ax.axvline(0, color='gray', linewidth=0.8, alpha=0.5)
ax.set_aspect('equal')
ax.set_title("Vector Space Geometry: Raw vs. Unit-Normalized Vectors", fontsize=12, pad=12)
ax.set_xlabel("Dimension 1 ($x_1$)")
ax.set_ylabel("Dimension 2 ($x_2$)")
ax.legend(loc="upper left", frameon=True, fontsize=9)
plt.tight_layout()
plt.show()

## 2. The Distance & Similarity Metric Trinity

In RAG retrieval, vector search engines compare a query embedding $\mathbf{q}$ against candidate chunk embeddings $\mathbf{c}$. Three primary metrics dominate vector retrieval:

### 1. Dot Product (Inner Product / IP)
$$\langle \mathbf{u}, \mathbf{v} \rangle = \mathbf{u} \cdot \mathbf{v} = \sum_{i=1}^D u_i v_i = \|\mathbf{u}\|_2 \|\mathbf{v}\|_2 \cos\theta$$
- **Property:** Measures both angular alignment *and* vector lengths.
- **Trap (The Length Bias Trap):** If chunks have variable lengths or unnormalized representations, a long chunk with an inflated norm can produce a higher dot product than a short, perfectly aligned chunk!

### 2. Cosine Similarity & Cosine Distance
$$S_C(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} = \cos\theta \in [-1.0, 1.0]$$
$$D_C(\mathbf{u}, \mathbf{v}) = 1.0 - S_C(\mathbf{u}, \mathbf{v}) \in [0.0, 2.0]$$
- **Property:** Pure angular orientation. Invariant to vector scaling ($S_C(\alpha \mathbf{u}, \beta \mathbf{v}) = S_C(\mathbf{u}, \mathbf{v})$ for any positive scalars $\alpha, \beta > 0$).
- **Use case:** Standard for natural language embeddings where text length should not skew semantic relevance.

### 3. Euclidean Distance ($L_2$ Distance)
$$D_{L_2}(\mathbf{u}, \mathbf{v}) = \|\mathbf{u} - \mathbf{v}\|_2 = \sqrt{\sum_{i=1}^D (u_i - v_i)^2}$$
- **Property:** True mathematical metric (satisfies identity of indiscernibles, symmetry, and triangle inequality).
- **Behavior:** Measures straight-line physical separation in $\mathbb{R}^D$. Highly sensitive to vector magnitude differences.

---

### Comparative Metric Properties

| Metric | Range | Scale Invariant? | Metric Space Axioms? | Hardware Execution |
|---|---|---|---|---|
| **Dot Product** | $(-\infty, +\infty)$ | ❌ No | ❌ No (not a distance metric) | Fastest (Single BLAS GEMM) |
| **Cosine Similarity** | $[-1.0, 1.0]$ | ✅ Yes | ❌ No ($1 - S_C$ violates triangle inequality) | Fast (requires norm division) |
| **Euclidean ($L_2$)** | $[0, +\infty)$ | ❌ No | ✅ Yes (complete metric space) | Slower (requires subtraction & sqrt) |

In the next code cell, we implement all three metrics from scratch and expose the **Length Bias Trap**.

In [ ]:
def dot_product(u: np.ndarray, v: np.ndarray) -> float:
    """Compute inner product between two vectors from first principles."""
    return float(np.sum(u * v))

def cosine_similarity(u: np.ndarray, v: np.ndarray, eps: float = 1e-12) -> float:
    """Compute cosine similarity with epsilon safeguard against division by zero."""
    norm_u = np.linalg.norm(u)
    norm_v = np.linalg.norm(v)
    if norm_u < eps or norm_v < eps:
        return 0.0
    return float(np.dot(u, v) / (norm_u * norm_v))

def cosine_distance(u: np.ndarray, v: np.ndarray) -> float:
    """Compute cosine distance: D_C = 1 - S_C."""
    return 1.0 - cosine_similarity(u, v)

def euclidean_distance(u: np.ndarray, v: np.ndarray) -> float:
    """Compute L2 Euclidean distance between two vectors."""
    return float(np.sqrt(np.sum((u - v) ** 2)))

# --- The Length Bias Trap Demonstration ---
# Scenario: A user asks a query q about 'Quantum Computing'.
# Candidate Chunk A is short, concise, and semantically aligned (cos_theta = 0.95), but norm = 2.0.
# Candidate Chunk B is a long sprawling document on 'General Science', semantically divergent (cos_theta = 0.50), but norm = 8.0.

q = np.array([1.0, 0.0], dtype=np.float64)       # Query direction
c_a = np.array([1.9, 0.62], dtype=np.float64)    # Chunk A (concise, high alignment)
c_b = np.array([4.0, 6.92], dtype=np.float64)    # Chunk B (long document, bloated norm)

dp_a = dot_product(q, c_a)
dp_b = dot_product(q, c_b)

cos_a = cosine_similarity(q, c_a)
cos_b = cosine_similarity(q, c_b)

euc_a = euclidean_distance(q, c_a)
euc_b = euclidean_distance(q, c_b)

print("=== Metric Comparison: Exposing the Length Bias Trap ===")
print(f"Chunk A (Concise) -> Dot Product: {dp_a:.2f} | Cosine Sim: {cos_a:.4f} | L2 Dist: {euc_a:.2f}")
print(f"Chunk B (Bloated) -> Dot Product: {dp_b:.2f} | Cosine Sim: {cos_b:.4f} | L2 Dist: {euc_b:.2f}")
print("-" * 55)
print(f"Dot Product Winner:     {'Chunk B (WRONG! Length skewed)' if dp_b > dp_a else 'Chunk A'}")
print(f"Cosine Sim Winner:      {'Chunk A (CORRECT! Pure semantic alignment)' if cos_a > cos_b else 'Chunk B'}")
print(f"Euclidean Dist Winner:  {'Chunk A (Closest in space)' if euc_a < euc_b else 'Chunk B'}")

## 3. The Fundamental Equivalence Theorem: Unit Sphere Normalization

Why do production vector databases (Milvus, Qdrant, ChromaDB, FAISS) insist on $L_2$ normalization? Because on the unit hypersphere $\mathbb{S}^{D-1}$, **all three metrics become algebraically equivalent monotonic transformations of each other!**

### Mathematical Derivation
Let $\mathbf{u}, \mathbf{v} \in \mathbb{R}^D$ be unit-normalized vectors such that $\|\mathbf{u}\|_2 = 1.0$ and $\|\mathbf{v}\|_2 = 1.0$.

Expand the squared Euclidean distance:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = (\mathbf{u} - \mathbf{v})^T (\mathbf{u} - \mathbf{v}) = \mathbf{u}^T \mathbf{u} + \mathbf{v}^T \mathbf{v} - 2 (\mathbf{u}^T \mathbf{v})$$

Substitute $\mathbf{u}^T \mathbf{u} = \|\mathbf{u}\|_2^2 = 1.0$ and $\mathbf{v}^T \mathbf{v} = \|\mathbf{v}\|_2^2 = 1.0$:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = 1.0 + 1.0 - 2 (\mathbf{u} \cdot \mathbf{v}) = 2 - 2 (\mathbf{u} \cdot \mathbf{v})$$

Since for unit vectors $\mathbf{u} \cdot \mathbf{v} = S_C(\mathbf{u}, \mathbf{v})$:
$$\|\mathbf{u} - \mathbf{v}\|_2^2 = 2 (1 - S_C(\mathbf{u}, \mathbf{v})) = 2 \cdot D_C(\mathbf{u}, \mathbf{v})$$

Taking the square root:
$$\mathbf{D_{L_2}(\mathbf{u}, \mathbf{v}) = \sqrt{2 \cdot (1 - S_C(\mathbf{u}, \mathbf{v}))} = \sqrt{2 \cdot D_C(\mathbf{u}, \mathbf{v})}}$$

### Crucial Production Implications
1. **Rank Preservation:** Maximizing Cosine Similarity is *strictly identical* to minimizing Euclidean distance and maximizing Dot Product:
   $$\arg\max_{\mathbf{v}} S_C(\mathbf{u}, \mathbf{v}) \equiv \arg\max_{\mathbf{v}} (\mathbf{u} \cdot \mathbf{v}) \equiv \arg\min_{\mathbf{v}} \|\mathbf{u} - \mathbf{v}\|_2$$
2. **Computational Speedup (GEMM):** Computing Euclidean distance requires subtractions, squarings, sum, and square roots ($O(4D)$ ops). Normalized Dot Product requires only a multiply-accumulate ($O(2D)$ ops).
3. Modern CPUs (AVX-512) and GPUs execute matrix multiplication (`GEMM`) at maximum hardware throughput. By pre-normalizing vectors once during ingestion, the retrieval engine only executes `matrix @ query`, achieving $2\times - 4\times$ higher query QPS!

In the next code cell, we formally test and verify this equivalence across 1,000 random vectors.

In [ ]:
# Generate a batch of random vectors in R^128
dim = 128
n_samples = 1000
raw_vectors = np.random.randn(n_samples, dim)

# L2-normalize all vectors onto the unit hypersphere
norms = np.linalg.norm(raw_vectors, axis=1, keepdims=True)
unit_vectors = raw_vectors / norms

# Select a reference query vector
query = unit_vectors[0]
candidates = unit_vectors[1:]

# Compute Cosine Similarity, Dot Product, and Euclidean Distance
cos_sims = np.dot(candidates, query)                     # Shape: (999,)
dot_prods = np.sum(candidates * query, axis=1)           # Shape: (999,)
l2_dists = np.linalg.norm(candidates - query, axis=1)    # Shape: (999,)

# Calculate theoretical L2 distance from Cosine Similarity
theoretical_l2 = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - cos_sims)))

# Assert algebraic identity across all pairs
max_error = np.max(np.abs(l2_dists - theoretical_l2))
assert np.allclose(l2_dists, theoretical_l2, atol=1e-7), f"Equivalence failed! Max error: {max_error}"

# Assert rank order consistency
rank_by_cos = np.argsort(-cos_sims)      # Descending order (highest similarity first)
rank_by_dot = np.argsort(-dot_prods)     # Descending order (highest dot product first)
rank_by_l2  = np.argsort(l2_dists)       # Ascending order (smallest distance first)

assert np.array_equal(rank_by_cos, rank_by_dot), "Cosine vs Dot rank mismatch!"
assert np.array_equal(rank_by_cos, rank_by_l2),  "Cosine vs L2 rank mismatch!"

print(f"Equivalence Theorem Verified across {n_samples-1} candidate vectors in R^{dim}:")
print(f"-> Maximum numerical divergence |L2 - sqrt(2*(1-cos))|: {max_error:.2e}")
print(f"-> Ranking consistency across all top candidates: 100.0% EXACT MATCH")
print(f"Top-3 Nearest Candidates Indices: {rank_by_cos[:3]} with similarities: {cos_sims[rank_by_cos[:3]].round(4)}")

## 4. Vectorized Batch Operations & Pairwise Gram Matrices

In production RAG pipelines, retrieval never processes vectors one by one with Python `for` loops. Instead, operations are batched as **BLAS Matrix Multiplications**:

Let the corpus of $N$ chunk embeddings be represented as matrix $\mathbf{X} \in \mathbb{R}^{N \times D}$ (where each row is an $L_2$-normalized embedding).  
Let a batch of $B$ user queries be $\mathbf{Q} \in \mathbb{R}^{B \times D}$.

The full batch similarity matrix $\mathbf{S} \in \mathbb{R}^{B \times N}$ is computed in a single operation:
$$\mathbf{S} = \mathbf{Q} \mathbf{X}^T$$

For internal document deduplication or clustering, the **Pairwise Gram Matrix** (all-pairs similarity) is:
$$\mathbf{G} = \mathbf{X} \mathbf{X}^T \in \mathbb{R}^{N \times N}$$

In the next cell, we implement a production-grade vectorized retriever with $O(1)$ Python dispatch and compare top-$k$ selection using `np.argpartition` vs `np.argsort`.

In [ ]:
def batch_l2_normalize(matrix: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    """Row-wise L2 normalization using vectorized broadcast."""
    norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    return matrix / np.maximum(norms, eps)

def vectorized_top_k_retrieval(
    query_vec: np.ndarray,
    corpus_matrix: np.ndarray,
    k: int = 3
) -> Tuple[np.ndarray, np.ndarray]:
    """
    Perform top-k similarity search using GEMM matrix-vector multiplication.
    Uses np.argpartition for O(N) top-k selection instead of O(N log N) full sort.
    """
    # Normalize inputs
    q_norm = query_vec / max(np.linalg.norm(query_vec), 1e-12)
    corpus_norm = batch_l2_normalize(corpus_matrix)
    
    # Batch cosine similarity via single dot product: S = X @ q
    scores = np.dot(corpus_norm, q_norm)
    
    # Fast top-k partition
    if k >= len(scores):
        top_k_indices = np.argsort(-scores)
    else:
        # argpartition finds the k largest elements in O(N) time
        partitioned_indices = np.argpartition(-scores, k)[:k]
        # Sort only the top k items in O(k log k) time
        sorted_top_k = np.argsort(-scores[partitioned_indices])
        top_k_indices = partitioned_indices[sorted_top_k]
        
    top_k_scores = scores[top_k_indices]
    return top_k_indices, top_k_scores

# Test with a mock corpus of 500 chunks in R^64
n_chunks = 500
test_corpus = np.random.randn(n_chunks, 64)
test_query = np.random.randn(64)

top_idx, top_scores = vectorized_top_k_retrieval(test_query, test_corpus, k=5)
print(f"Top 5 Retrieved Chunk IDs: {top_idx}")
print(f"Top 5 Cosine Scores:       {top_scores.round(4)}")

## 5. Semantic Topology: Realistic Topic Clusters & Gram Heatmap

To observe how semantic distance manifests in real language without calling external APIs or downloading multi-gigabyte models, we construct a **deterministic semantic projection model**.

We define 8 realistic document snippets across 4 distinct conceptual domains:
1. **Artificial Intelligence & Machine Learning**
2. **Astrophysics & Space Exploration**
3. **Financial Markets & Economics**
4. **Culinary Arts & Gastronomy**

Our offline semantic projector maps text into domain feature spaces, producing realistic embeddings that exhibit true cross-domain orthogonality and within-domain clustering.

In the following code cell, we compute the full $8 \times 8$ Pairwise Gram Matrix and visualize the block-diagonal structure.

In [ ]:
# 8 Realistic domain chunks (2 chunks per domain)
documents = [
    # Domain 1: AI / Machine Learning
    "Transformer architectures rely on self-attention mechanisms to learn contextual representations.",
    "Deep neural networks require backpropagation and gradient descent for parameter optimization.",
    
    # Domain 2: Astrophysics / Space
    "Gravitational waves propagate through spacetime following massive stellar black hole mergers.",
    "The James Webb Space Telescope observes high-redshift galaxies from the Sun-Earth L2 Lagrange point.",
    
    # Domain 3: Financial Markets
    "Central banks adjust interest rates to manage inflationary pressure and balance liquidity.",
    "Equity portfolio risk can be hedged using index futures and option derivatives contracts.",
    
    # Domain 4: Culinary Arts
    "Caramelization and the Maillard reaction develop complex flavor profiles when searing meat.",
    "Sourdough fermentation relies on wild yeast and lactobacillus bacteria to produce lactic acid."
]

doc_labels = [
    "AI: Attention Mechanisms",
    "AI: Gradient Descent",
    "Astro: Gravitational Waves",
    "Astro: James Webb Telescope",
    "Finance: Interest Rates",
    "Finance: Derivatives Hedging",
    "Culinary: Maillard Reaction",
    "Culinary: Sourdough Ferment"
]

# Domain semantic vocabulary anchors
vocabulary = {
    "ai": ["transformer", "neural", "attention", "gradient", "descent", "backpropagation", "learning", "representations", "optimization"],
    "astro": ["gravitational", "spacetime", "black", "hole", "telescope", "galaxies", "redshift", "lagrange", "stellar"],
    "finance": ["central", "banks", "interest", "rates", "inflationary", "liquidity", "equity", "derivatives", "hedged"],
    "culinary": ["caramelization", "maillard", "flavor", "searing", "sourdough", "fermentation", "yeast", "lactic", "acid"]
}

def deterministic_semantic_embed(text: str, dim: int = 32) -> np.ndarray:
    """
    Deterministic semantic projection: creates a 32-dim dense embedding
    by projecting text onto semantic domain axes + deterministic hash variance.
    100% offline, reproducible, zero external downloads.
    """
    tokens = text.lower().replace(".", "").replace(",", "").split()
    # 4 domain core energies
    domain_weights = np.zeros(4, dtype=np.float64)
    for idx, (domain, words) in enumerate(vocabulary.items()):
        matches = sum(1 for t in tokens if t in words)
        domain_weights[idx] = matches + 0.05  # slight baseline
        
    # Project 4 domain energies into 32 dimensions via fixed orthogonal projection
    rng = np.random.RandomState(1337)
    proj_matrix = rng.randn(4, dim)
    # Add subtle hash-based word perturbation for uniqueness
    word_hash_seed = sum(ord(c) for c in text) % 10000
    noise_rng = np.random.RandomState(word_hash_seed)
    noise = noise_rng.randn(dim) * 0.08
    
    vec = domain_weights @ proj_matrix + noise
    # L2 normalize
    return vec / np.linalg.norm(vec)

# Generate embeddings for all documents
doc_embeddings = np.array([deterministic_semantic_embed(doc) for doc in documents])

# Compute 8x8 Pairwise Gram Matrix (Cosine Similarity Matrix)
gram_matrix = np.dot(doc_embeddings, doc_embeddings.T)

# Plot Heatmap
fig, ax = plt.subplots(figsize=(7.5, 6))
cax = ax.matshow(gram_matrix, cmap="Blues", vmin=0.0, vmax=1.0)
fig.colorbar(cax, fraction=0.046, pad=0.04, label="Cosine Similarity")

ax.set_xticks(range(len(doc_labels)))
ax.set_yticks(range(len(doc_labels)))
ax.set_xticklabels(doc_labels, rotation=45, ha="left", fontsize=8.5)
ax.set_yticklabels(doc_labels, fontsize=8.5)

# Annotate cells with values
for i in range(len(doc_labels)):
    for j in range(len(doc_labels)):
        val = gram_matrix[i, j]
        color = "white" if val > 0.6 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", color=color, fontsize=8, fontweight="bold" if i==j or (i//2 == j//2) else "normal")

ax.set_title("Pairwise Semantic Similarity Heatmap (Gram Matrix)", fontsize=11, pad=20)
plt.tight_layout()
plt.show()

# Verify within-cluster similarity is significantly higher than cross-cluster similarity
within_cluster_sims = [gram_matrix[0, 1], gram_matrix[2, 3], gram_matrix[4, 5], gram_matrix[6, 7]]
cross_cluster_sims = [gram_matrix[0, 2], gram_matrix[1, 4], gram_matrix[3, 6], gram_matrix[5, 7]]
print(f"Average Intra-Domain Similarity: {np.mean(within_cluster_sims):.3f}")
print(f"Average Cross-Domain Similarity: {np.mean(cross_cluster_sims):.3f}")
assert np.mean(within_cluster_sims) > np.mean(cross_cluster_sims) + 0.4, 'Clustering separation failure!'


## 6. The Curse of Dimensionality & Distance Concentration

Modern embedding models operate in very high-dimensional spaces:
- OpenAI `text-embedding-3-small`: $D = 1536$
- OpenAI `text-embedding-3-large`: $D = 3072$
- BAAI `bge-large-en-v1.5`: $D = 1024$
- Sentence-Transformers `all-MiniLM-L6-v2`: $D = 384$

Human spatial intuition is formed in 2D and 3D space, which completely fails in high dimensions due to three counter-intuitive geometric phenomena:

### 1. Orthogonality of Random Vectors
In high dimensions, **almost all randomly chosen vectors are nearly orthogonal!**  
For two random unit vectors $\mathbf{u}, \mathbf{v} \sim \mathcal{N}(0, \frac{1}{D} \mathbf{I})$:
$$\mathbb{E}[\mathbf{u} \cdot \mathbf{v}] = 0, \quad \text{Var}(\mathbf{u} \cdot \mathbf{v}) = \frac{1}{D}$$
As $D \to \infty$, the standard deviation shrinks as $1/\sqrt{D}$. In $D = 1536$, the standard deviation is only $1/\sqrt{1536} \approx 0.025$. Random vectors have cosine similarities tightly concentrated in $[-0.05, +0.05]$.

### 2. Distance Concentration Phenomenon (Beyer et al., 1999)
As dimensionality $D$ increases, the difference between the distance to the nearest neighbor ($D_{\min}$) and the distance to the farthest neighbor ($D_{\max}$) relative to $D_{\min}$ approaches zero:
$$\lim_{D \to \infty} \frac{D_{\max} - D_{\min}}{D_{\min}} = 0$$
In high dimensions, every point is almost equidistant from every other point!

### 3. Hypersphere Surface Volume Concentration
In a $D$-dimensional unit sphere, virtually $100\%$ of the volume is concentrated in a microscopic shell right beneath the surface:
$$\frac{V(r = 1 - \epsilon)}{V(r = 1)} = (1 - \epsilon)^D \xrightarrow{D \to \infty} 0$$
For $D = 1536$ and $\epsilon = 0.01$ (the outer $1\%$ shell), $(0.99)^{1536} \approx 1.8 \times 10^{-7}$. That is, **$99.99998\%$ of all volume resides in the outer $1\%$ shell!**

In the next code cell, we simulate and measure distance concentration across dimensions $D \in [2, 16, 128, 768, 1536]$.

In [ ]:
dimensions = [2, 16, 128, 768, 1536]
n_pairs = 2000

concentration_stats = {}

for d in dimensions:
    # Generate random points from standard Gaussian
    points = np.random.randn(n_pairs, d)
    # L2 normalize onto unit sphere
    unit_pts = points / np.linalg.norm(points, axis=1, keepdims=True)
    
    # Pick first 500 vectors and another 500 vectors to compute pairwise distances
    set_a = unit_pts[:500]
    set_b = unit_pts[500:1000]
    
    # Pairwise cosine similarities and L2 distances
    cos_sims = np.sum(set_a * set_b, axis=1)
    l2_dists = np.linalg.norm(set_a - set_b, axis=1)
    
    # Relative contrast: (D_max - D_min) / D_min
    d_max = np.max(l2_dists)
    d_min = np.min(l2_dists)
    rel_contrast = (d_max - d_min) / d_min
    
    concentration_stats[d] = {
        "cos_mean": float(np.mean(cos_sims)),
        "cos_std": float(np.std(cos_sims)),
        "l2_mean": float(np.mean(l2_dists)),
        "l2_std": float(np.std(l2_dists)),
        "rel_contrast": float(rel_contrast),
        "cos_sims": cos_sims,
        "l2_dists": l2_dists
    }

print(f"{'Dim (D)':<10} | {'Cosine Std':<12} | {'L2 Mean':<10} | {'L2 Std':<10} | {'Relative Contrast':<18}")
print("-" * 70)
for d in dimensions:
    stats = concentration_stats[d]
    print(f"{d:<10} | {stats['cos_std']:<12.4f} | {stats['l2_mean']:<10.4f} | {stats['l2_std']:<10.4f} | {stats['rel_contrast']:<18.4f}")

### Visualizing Distance Concentration Across Dimensions

The next cell plots empirical probability distributions of **Cosine Similarity** and **Euclidean Distance** across dimensions $D=2, 16, 128, 768, 1536$.

Observe how:
1. In $D = 2$, cosine similarity is broadly distributed across the entire range $[-1.0, 1.0]$.
2. In $D = 1536$, the distribution collapses into a razor-thin needle centered exactly at $\cos\theta = 0.0$ and Euclidean distance $D_{L_2} = \sqrt{2} \approx 1.414$.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

colors = {2: "#94a3b8", 16: "#60a5fa", 128: "#3b82f6", 768: "#1d4ed8", 1536: "#ef4444"}

for d in dimensions:
    sims = concentration_stats[d]["cos_sims"]
    dists = concentration_stats[d]["l2_dists"]
    
    ax1.hist(sims, bins=40, density=True, histtype='step', linewidth=2.0,
             color=colors[d], label=f"D = {d} (σ={concentration_stats[d]['cos_std']:.3f})")
    ax2.hist(dists, bins=40, density=True, histtype='step', linewidth=2.0,
             color=colors[d], label=f"D = {d} (σ={concentration_stats[d]['l2_std']:.3f})")

ax1.set_title("Cosine Similarity Distribution Collapse (θ → 90°)", fontsize=11)
ax1.set_xlabel("Cosine Similarity ($S_C$)")
ax1.set_ylabel("Probability Density")
ax1.set_xlim(-1.05, 1.05)
ax1.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax1.legend(loc="upper right", fontsize=8.5)

ax2.set_title(r"Euclidean Distance Concentration ($D_{L_2} \to \sqrt{2} \approx 1.414$)", fontsize=11)
ax2.set_xlabel(r"Euclidean Distance ($D_{L_2}$)")
ax2.set_ylabel("Probability Density")
ax2.axvline(math.sqrt(2.0), color='#ef4444', linestyle='--', alpha=0.7, label=r"Theoretical $\sqrt{2} \approx 1.414$")
ax2.legend(loc="upper left", fontsize=8.5)

plt.tight_layout()
plt.show()

## 7. Dimensional Collapse & Embedding Anisotropy ("The Narrow Cone Problem")

While theoretical random vectors are uniformly isotropic (occupying all directions of the unit sphere equally), **empirical transformer embeddings suffer from severe anisotropy (dimensional collapse)**.

### Why Does Anisotropy Happen in Transformer Embeddings?
Research (Ethayarajh 2019, Gao et al. 2021) shows that contextual embeddings from BERT, RoBERTa, and early GPT models collapse into a **narrow, high-density cone**:
1. **Word Frequency Bias:** High-frequency functional words dominate token representations and pull token embeddings toward a dominant shared direction vector $\boldsymbol{\mu}$.
2. **Dominant Singular Values:** The embedding covariance matrix has a few massive singular values (eigenvalues), while the remaining 95% of spatial dimensions carry near-zero variance.
3. **Symptom:** Completely unrelated texts exhibit an average cosine similarity of $+0.60$ to $+0.80$, rather than the theoretical expectation of $0.0$.

### Measuring and Mitigating Anisotropy
1. **Singular Value Decomposition (SVD):** Compute $\mathbf{X} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$. In isotropic space, singular values are uniform. In collapsed space, the top singular value $\sigma_1$ dominates.
2. **Mean-Centering:** Shift embeddings by their global centroid:
   $$\tilde{\mathbf{x}} = \mathbf{x} - \boldsymbol{\mu}, \quad \text{where } \boldsymbol{\mu} = \frac{1}{N} \sum_{i=1}^N \mathbf{x}_i$$
   Then re-normalize onto the unit sphere. This centers the distribution at the origin and restores angular dynamic range!

In the next code cell, we simulate an anisotropic embedding space, inspect its singular value spectrum, apply mean-centering, and visualize the geometric cone vs. isotropic sphere in 3D.

In [ ]:
# 1. Simulate an Anisotropic Space (Narrow Cone)
n_samples = 400
d_vis = 3

# Define a dominant cone direction vector (shared bias vector)
cone_center = np.array([1.0, 1.0, 1.0])
cone_center = cone_center / np.linalg.norm(cone_center)

# Generate vectors tightly clustered around the cone direction
noise = np.random.randn(n_samples, d_vis) * 0.22
anisotropic_raw = cone_center + noise
anisotropic_pts = anisotropic_raw / np.linalg.norm(anisotropic_raw, axis=1, keepdims=True)

# 2. Simulate an Isotropic Space (Uniform Sphere)
uniform_raw = np.random.randn(n_samples, d_vis)
isotropic_pts = uniform_raw / np.linalg.norm(uniform_raw, axis=1, keepdims=True)

# 3. Mitigation: Apply Mean-Centering to Anisotropic Points
mu = np.mean(anisotropic_pts, axis=0)
centered_raw = anisotropic_pts - mu
centered_pts = centered_raw / np.linalg.norm(centered_raw, axis=1, keepdims=True)

# Compare SVD singular values
_, s_aniso, _ = np.linalg.svd(anisotropic_pts, full_matrices=False)
_, s_iso, _ = np.linalg.svd(isotropic_pts, full_matrices=False)
_, s_centered, _ = np.linalg.svd(centered_pts, full_matrices=False)

# Compute average pairwise cosine similarity
sample_pairs = 1000
idx_i = np.random.randint(0, n_samples, sample_pairs)
idx_j = np.random.randint(0, n_samples, sample_pairs)

mean_cos_aniso = float(np.mean(np.sum(anisotropic_pts[idx_i] * anisotropic_pts[idx_j], axis=1)))
mean_cos_iso = float(np.mean(np.sum(isotropic_pts[idx_i] * isotropic_pts[idx_j], axis=1)))
mean_cos_centered = float(np.mean(np.sum(centered_pts[idx_i] * centered_pts[idx_j], axis=1)))

print("=== Anisotropy Diagnostic Metrics ===")
print(f"Anisotropic (Collapsed Cone)   -> Mean Cosine: {mean_cos_aniso:.4f} | Singular Values: {s_aniso.round(2)}")
print(f"Isotropic (Uniform Sphere)     -> Mean Cosine: {mean_cos_iso:.4f} | Singular Values: {s_iso.round(2)}")
print(f"Mean-Centered & Re-normalized  -> Mean Cosine: {mean_cos_centered:.4f} | Singular Values: {s_centered.round(2)}")

# 3D Geometric Plot
fig = plt.figure(figsize=(12, 5))

# Subplot 1: Anisotropic Cone
ax1 = fig.add_subplot(1, 2, 1, projection='3d')
ax1.scatter(anisotropic_pts[:, 0], anisotropic_pts[:, 1], anisotropic_pts[:, 2],
            c='#ef4444', alpha=0.6, s=18, label="Collapsed Embeddings")
# Plot wireframe sphere
u_wire, v_wire = np.mgrid[0:2*np.pi:20j, 0:np.pi:10j]
x_wire = np.cos(u_wire) * np.sin(v_wire)
y_wire = np.sin(u_wire) * np.sin(v_wire)
z_wire = np.cos(v_wire)
ax1.plot_wireframe(x_wire, y_wire, z_wire, color="gray", alpha=0.15)
ax1.set_title(f"Anisotropic Space (Collapsed Cone)\nAvg Cosine = {mean_cos_aniso:.2f}", fontsize=10)
ax1.set_box_aspect([1, 1, 1])

# Subplot 2: Restored Isotropic Space (Mean Centered)
ax2 = fig.add_subplot(1, 2, 2, projection='3d')
ax2.scatter(centered_pts[:, 0], centered_pts[:, 1], centered_pts[:, 2],
            c='#2563eb', alpha=0.6, s=18, label="Mean-Centered Embeddings")
ax2.plot_wireframe(x_wire, y_wire, z_wire, color="gray", alpha=0.15)
ax2.set_title(f"Restored Space (After Mean Centering)\nAvg Cosine = {mean_cos_centered:.2f}", fontsize=10)
ax2.set_box_aspect([1, 1, 1])

plt.tight_layout()
plt.show()

## 8. Notebook Verification & Automated Validation Suite

To ensure absolute pedagogical and mathematical correctness, this final cell executes self-testing assertions covering:
1. Metric calculations against analytical mathematical targets.
2. Zero division handling in cosine similarity.
3. Strict rank equivalence between Cosine, Dot Product, and Euclidean metrics under $L_2$ normalization.
4. Scale invariance of Cosine Similarity.
5. SVD spectrum decay and anisotropic mitigation assertions.

In [ ]:
# 1. Analytical Metric Verification
vec_x = np.array([1.0, 0.0])
vec_y = np.array([0.0, 1.0])
vec_opp = np.array([-1.0, 0.0])

assert np.isclose(cosine_similarity(vec_x, vec_y), 0.0), "Orthogonal cosine error"
assert np.isclose(cosine_similarity(vec_x, vec_x), 1.0), "Identical cosine error"
assert np.isclose(cosine_similarity(vec_x, vec_opp), -1.0), "Opposite cosine error"
assert np.isclose(euclidean_distance(vec_x, vec_y), math.sqrt(2.0)), "Orthogonal L2 error"

# 2. Scale Invariance Check
scaled_x = vec_x * 42.0
scaled_y = vec_y * 0.001
assert np.isclose(cosine_similarity(scaled_x, scaled_y), 0.0), "Scale invariance failed"

# 3. Safe zero vector check
zero_vec = np.zeros(2)
assert cosine_similarity(zero_vec, vec_x) == 0.0, "Zero division safeguard failed"

# 4. Equivalence check across dimensional scaling
for test_dim in [4, 16, 64, 256]:
    a = np.random.randn(test_dim)
    b = np.random.randn(test_dim)
    a_u = a / np.linalg.norm(a)
    b_u = b / np.linalg.norm(b)
    
    expected_l2 = math.sqrt(max(0.0, 2.0 * (1.0 - np.dot(a_u, b_u))))
    actual_l2 = np.linalg.norm(a_u - b_u)
    assert np.isclose(expected_l2, actual_l2, atol=1e-6), f"Equivalence failed at dim {test_dim}"

# 5. Mean-centering anisotropy reduction check
assert abs(mean_cos_centered) < abs(mean_cos_aniso) * 0.25, "Mean centering failed to collapse anisotropy"

print("==================================================================")
print("  ALL 5 VERIFICATION SUITES PASSED SUCCESSFULLY (100% ACCURACY)  ")
print("==================================================================")
print("Track 02.1 is validated and ready for indexing and ANN graphs (Track 02.2).")